In [ ]:
# %%
import sagemaker

import boto3
from botocore.exceptions import ClientError
import ast



In [ ]:
# secret shhhhh
def get_secret(session):
    """
    Retreive the GitLab secret from AWS SecretsManager

    Returns: Dict, with keys "username" and "password" for GitLab
    """
    secret_name = "uwgitlab/access-token"
    region_name = "ca-central-1"

    # Create a Secrets Manager client
    client = session.client(
        service_name='secretsmanager',
        region_name=region_name
    )

    try:
        get_secret_value_response = client.get_secret_value(
            SecretId=secret_name
        )
    except ClientError as e:
        # For a list of exceptions thrown, see
        # https://docs.aws.amazon.com/secretsmanager/latest/apireference/API_GetSecretValue.html
        raise e

    # Decrypts secret using the associated KMS key.
    secret = get_secret_value_response['SecretString']

    # Convert raw string into Python dict
    secret = ast.literal_eval(secret)
    return secret

# hf secret shhhhh
def get_secret_hf(session):

    secret_name = "hf-access-token"
    region_name = "ca-central-1"

    # Create a Secrets Manager client
    client = session.client(
        service_name='secretsmanager',
        region_name=region_name
    )
    
    try:
        get_secret_value_response = client.get_secret_value(
            SecretId=secret_name
        )
    except ClientError as e:
        # For a list of exceptions thrown, see
        # https://docs.aws.amazon.com/secretsmanager/latest/apireference/API_GetSecretValue.html
        raise e

    # Decrypts secret using the associated KMS key.
    secret = get_secret_value_response['SecretString']

    # Your code goes here.
    return ast.literal_eval(secret)['hf-access-token']

# Create a Secrets Manager session
session = boto3.session.Session(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN
)

git_cred = get_secret(session)
hf_token = get_secret_hf(session)

In [ ]:
# Git config
git_config = {
    'repo': ,
    'username': ,
    'password': 
}

# Script CLI args
hyperparameters = {
    'epochs': 1, 
    # 'lr': 1e-4,
    "seq_len": 1024,
    "adam_beta1": 0.9,
    "adam_beta2": 0.95,
    "learning_rate": 5e-5,
    "max_grad_norm":1.0,
    "gradient_accumulation_steps": 2,
    "gradient_checkpointing": True,
    "deepspeed": 'deepspeed_config.json',
    "output_dir": "/opt/ml/output",
    "per_device_train_batch_size":2,
    "num_train_epochs":1 ,
    "fp16":True,
    "logging_steps":10,
    "save_total_limit": 1,
    'hf_token': hf_token,
    "training_script": 'LLAMA_KA_Pretraining.py',
    # "metric_for_best_model": 'train_loss'
    # "load_best_model_at_end": True,
    # "per_device_eval_batch_size":2,
    # "evaluation_strategy": "steps",
    # "eval_steps": 2,
    # "save_steps": 4,
    
}

SAGEMAKER_EXECUTION_ROLE = "arn:aws:iam::506487962374:role/SageMaker-AI-Execution-Role"
ECR_IMAGE_URI = "506487962374.dkr.ecr.ca-central-1.amazonaws.com/ka-llm-training:deepspeed-pretraining"

settings=sagemaker.session_settings.SessionSettings()

sagemaker_session = sagemaker.session.Session(boto_session=session, settings=settings)

estimator_args = {
    # Job config
    "base_job_name": "ka-pretraining",
    "role": SAGEMAKER_EXECUTION_ROLE,
    "output_path": "s3://eko-ekoka-ai-project/output",
    "image_uri": ECR_IMAGE_URI,
    # Instance config
    "instance_count": 1,
    "instance_type": "ml.p3.16xlarge",
    # Script config
    "entry_point": "launch_deepspeed_pt.py",
    "git_config": git_config,
    "source_dir": './scripts',
    "hyperparameters": hyperparameters,
    # Security
    "subnets": ['subnet-0ba981d4c6314f9a9'],
    "security_group_ids": ['sg-04645476ef00795e3'],
    "encrypt_inter_container_traffic": True,
    "sagemaker_session": sagemaker_session,
    "volume_size": 500,
}

pytorch_estimator = sagemaker.estimator.Estimator(**estimator_args, tags=[{"Key":'uw-ai-fine-tuning', "Value":'fine-tune'}])

In [ ]:
# Channel - S3 URL pairs for input data to map to within the training instance
input_data = {
    "train": "s3://eko-ekoka-ai-project/data/pt_dataset_chat",
    #"test": "s3://billsum-prototype/data/billsum_for_gpt_2048/test",
    #"tokenizer": "s3://billsum-prototype/data/tokenizer",
    #"model": "s3://billsum-prototype/data/model"
}

# %%
pytorch_estimator.fit(inputs=input_data)

In [ ]:
# s3 = boto3.client('s3')


# s3.upload_file(

#         '../scripts/deepspeed_config.json',

#         'eko-ekoka-ai-project',

#         "deepspeed_config.json",

#         ExtraArgs= {'ServerSideEncryption' : 'aws:kms'}  

#        )